[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/experiments/datasets_and_experiments_quickstart.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
        <br>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">快速入门：数据集与实验</h1>

Phoenix 帮助你在 AI 与 LLM 应用上运行实验，评估并迭代改进性能。本快速入门展示如何快速开始。

## 环境准备

安装 Phoenix。

In [ ]:
!pip install "arize-phoenix[evals]" openai

启动 Phoenix。

In [ ]:
import phoenix as px

px.launch_app().view()

设置 OpenAI API 密钥。

In [ ]:
import os
from getpass import getpass

import openai

if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥：")
openai.api_key = openai_api_key
os.environ["OPENAI_API_KEY"] = openai_api_key

初始化 Phoenix Client。它是与 Phoenix API 交互的主要入口，可独立于 Phoenix 本体安装。

In [ ]:
from phoenix.client import AsyncClient

px_client = AsyncClient()

## 数据集

上传数据集。

In [ ]:
dataset = await px_client.datasets.create_dataset(
    name="experiment-quickstart-dataset",
    inputs=[{"question": "What is Paul Graham known for?"}],
    outputs=[{"answer": "Co-founding Y Combinator and writing on startups and techology."}],
    metadata=[{"topic": "tech"}],
)

## 任务

创建待评估的任务。

In [ ]:
from typing import Any

from openai import OpenAI

openai_client = OpenAI()

task_prompt_template = "用几个词回答：{question}"


def task(input: Any) -> str:
    question = input["question"]
    message_content = task_prompt_template.format(question=question)
    response = openai_client.chat.completions.create(
        model="gpt-4o", messages=[{"role": "user", "content": message_content}]
    )
    return response.choices[0].message.content or ""

## 评估器

可以将评估器定义为函数。若函数只有一个参数，将使用任务输出调用它；否则评估器函数可以使用以下参数的任意组合：

- `input`：数据集示例的输入字段
- `output`：任务输出
- `expected`：数据集示例的预期输出或参考输出
- `reference`：`expected` 的别名
- `metadata`：数据集示例关联的元数据


In [ ]:
def contains_keyword(output: str) -> float:
    keywords = ["Y Combinator", "YC"]
    output_lower = output.lower()
    return 1.0 if any(keyword.lower() in output_lower for keyword in keywords) else 0.0

In [ ]:
from typing import Any, Dict


def jaccard_similarity(output: str, expected: Dict[str, Any]) -> float:
    # https://en.wikipedia.org/wiki/Jaccard_index
    actual_words = set(output.lower().split(" "))
    expected_words = set(expected["answer"].lower().split(" "))
    words_in_common = actual_words.intersection(expected_words)
    all_words = actual_words.union(expected_words)
    return len(words_in_common) / len(all_words)

也可以使用 LLM。

In [ ]:
from phoenix.client.experiments import create_evaluator

eval_prompt_template = """
根据 QUESTION 和 REFERENCE_ANSWER，判断 ANSWER 是否准确。
仅输出单个词：accurate 或 inaccurate。

QUESTION: {question}

REFERENCE_ANSWER: {reference_answer}

ANSWER: {answer}

ACCURACY (accurate / inaccurate):
"""


@create_evaluator(kind="llm")  # 需要此装饰器，否则 kind 默认为 "code"
def accuracy(input: Dict[str, Any], output: str, expected: Dict[str, Any]) -> float:
    message_content = eval_prompt_template.format(
        question=input["question"], reference_answer=expected["answer"], answer=output
    )
    response = openai_client.chat.completions.create(
        model="gpt-4o", messages=[{"role": "user", "content": message_content}]
    )
    response_message_content = response.choices[0].message.content.lower().strip()
    return 1.0 if response_message_content == "accurate" else 0.0

## 实验

运行实验并评估结果。

In [ ]:
experiment = await px_client.experiments.run_experiment(
    dataset=dataset,
    task=task,
    experiment_name="initial-experiment",
    evaluators=[contains_keyword, accuracy],
)

运行完成后再增加评估器。

In [ ]:
experiment = await px_client.experiments.evaluate_experiment(
    experiment=experiment, evaluators=[contains_keyword, jaccard_similarity]
)

In [ ]:
def show_evaluation_summary(exp: Any):
    contains_keyword_scores = [
        run.result["score"]
        for run in exp["evaluation_runs"]
        if run.result and "score" in run.result and run.name == "contains_keyword"
    ]
    jaccard_scores = [
        run.result["score"]
        for run in exp["evaluation_runs"]
        if run.result and "score" in run.result and run.name == "jaccard_similarity"
    ]
    accuracy_scores = [
        run.result["score"]
        for run in exp["evaluation_runs"]
        if run.result and "score" in run.result and run.name == "accuracy"
    ]

    print("📊 评估结果：")
    if contains_keyword_scores:
        avg_contains = sum(contains_keyword_scores) / len(contains_keyword_scores)
        print(f"  包含关键词： {avg_contains:.3f} (n={len(contains_keyword_scores)})")

    if accuracy_scores:
        avg_accuracy = sum(accuracy_scores) / len(accuracy_scores)
        print(f"  Accuracy: {avg_accuracy:.3f} (n={len(accuracy_scores)})")

    if jaccard_scores:
        avg_jaccard = sum(jaccard_scores) / len(jaccard_scores)
        print(f"  Jaccard 相似度： {avg_jaccard:.3f} (n={len(jaccard_scores)})")


show_evaluation_summary(experiment)

持续迭代 🚀